In [ ]:
# import libraries

import pandas as pd
import numpy as np
import seaborn as sns
sns.set(color_codes=True)
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
import math 

#update this path with the path where you stored the datasets
data_path = "/data/test_newrepo" 

In [ ]:
#Load LUTs

lut_name_soft = "soft_lut"
lut_name_medium = "medium_lut"
lut_name_hard = "hard_lut"

lut_soft = np.load(data_path+'/LUT_'+lut_name_soft+'.npy')
lut_medium = np.load(data_path+'/LUT_'+lut_name_medium+'.npy')
lut_hard = np.load(data_path+'/LUT_'+lut_name_hard+'.npy')

In [ ]:
# Save the LUT to plot it with external scripts
import pickle
if False:
    with open(prefix+"/data/lut_hard_Y1.pkl", "wb") as f:
        pickle.dump(lut_hard[:,4], f)


In [ ]:
#nside = 32

run_name_test = "dataset"
file_name_test = "run57_mix_mega_shared"

file_path = data_path+'/'+file_name_test+'_dataset.pkl'

# Load the array from the pickle file
with open(file_path, 'rb') as file:
    loaded_array_test = pickle.load(file)


In [ ]:
loaded_array_test.shape

In [ ]:
# Filter the dataset by spectral model and flux
# for soft spectra use flux = [14,16] and filter the spectra to contain the string "230"
# for medium spectra use flux = [1,3] and filter the spectra to contain the string "699"
# for hard spectra use flux = [1,1,6] and filter the spectra to contain the string "1500"

filter_flux = 1
filter_spectra = 1
filter_theta=0
filter_phi=0

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_flux == 1:
    for grb in loaded_array_test:
        if grb['flux'] >14 and grb['flux'] <= 16:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

print(len(loaded_array_test))
filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_spectra==1:
    for grb in loaded_array_test:
        if  "230" in grb['spectrum']: #["Band 10 10000 -1.9 -3.7 230","Band 10 10000 -1 -2.3 699.9","Comptonized 10 10000 -0.5 1500"]
        # if  grb['spectra'] == 'soft':
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

print(len(loaded_array_test))
count = 0
filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

if filter_theta==1:
    print("filter spectra")
    for grb in loaded_array_test:
        if float(grb['coord'][0])>50 and float(grb['coord'][0])<130:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

count = 0
filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

if filter_phi==1:
    print("filter spectra")
    for grb in loaded_array_test:
        if float(grb['coord'][1])>100 and float(grb['coord'][1])<170:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

In [ ]:
test_dataset = loaded_array_test

In [ ]:
test_dataset.shape

In [ ]:
def get_radians(coords):
    # Unpack the list of (theta, phi) pairs
    theta, phi = zip(*coords)
    
    # Convert to numpy arrays
    theta = np.array(theta)
    phi = np.array(phi)
    
    # Wrap phi values >180 into the range (-180, 180]
    mask = phi > 180
    phi[mask] -= 360
    
    # Convert theta into colatitude (90 - theta)
    theta = 90 - theta

    # Return values in radians
    return np.radians(theta), np.radians(phi)

def calculate_chi_squared_optimized(s, b, m):
    """
    Compute the chi-squared value for each position in the grid in an optimized way.

    Parameters:
        s: array of shape (6,) with observed counts (s(j)).
        b: array of shape (12,) with background counts (b(j)).
        m: array of shape (12, 41168) with model counts (m(j, i)).

    Returns:
        chi_squared: array of shape (41168,) with the chi-squared value for each position i.
    """
    # Use only the first 6 detectors
    indices = np.arange(6)
    
    # Extract model counts for these detectors
    m_subset = m[:, indices]  # Shape: (41168, 6)

    # Compute numerator and denominator for normalization factor f_i
    with np.errstate(divide='ignore', invalid='ignore'):
        numerator = np.sum(m_subset * (s[indices] - b[indices]) / s[indices], axis=1)
        denominator = np.sum((m_subset**2) / s[indices], axis=1)

        # Avoid division by zero
        f_i = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator != 0)

        # Expand f_i for broadcasting
        f_i_expanded = f_i[:, np.newaxis]  # Shape: (41168, 1)

        # Compute chi-squared terms
        chi_numerator = (s[indices] - b[indices] - f_i_expanded * m_subset)**2
        chi_denominator = b[indices] + f_i_expanded * m_subset

        # Safe division for chi-squared elements
        chi_squared_elements = np.divide(
            chi_numerator, chi_denominator,
            out=np.full_like(chi_numerator, np.finfo(np.float64).max),
            where=chi_denominator != 0
        )

    # Sum over detectors to obtain chi^2 per direction
    chi_squared = np.sum(chi_squared_elements, axis=1)

    return chi_squared


def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg

def diff_phi(a1, a2):
    diff = abs(a1 - a2)
    if diff > 180:
        diff = 360 - diff
    return diff

In [ ]:
def find_closest_pixel(theta_real, phi_real, best_lut):
    # Convert input angles from degrees to radians
    theta_real_rad = np.deg2rad(theta_real)
    phi_real_rad = np.deg2rad(phi_real)
    
    # Extract θ and φ from best_lut (last two columns, in degrees) and convert to radians
    theta_vals_m = np.deg2rad(best_lut[:, -2])
    phi_vals_m = np.deg2rad(best_lut[:, -1])
    
    # Compute the cosine of the spherical angular distance
    cos_dist = (
        np.sin(theta_real_rad) * np.sin(theta_vals_m) * np.cos(phi_real_rad - phi_vals_m)
        + np.cos(theta_real_rad) * np.cos(theta_vals_m)
    )

    # Compute the angular distance (in radians)
    angular_distance = np.arccos(cos_dist)
    
    # Find the index of the minimum distance
    min_idx = np.argmin(angular_distance)
    
    # The index of the maximum cosine corresponds to the minimum distance
    # min_idx = np.argmax(cos_dist)
    return min_idx

In [ ]:
def analyze_grbs(test_dataset,spectral_model):

    results = []
    spectra_fitted = 0
    good_spectra_fit_distances = []
    bad_spectra_fit_distances=[]
    
    count = 0
    for grb in test_dataset:
    
        #if count == 10:
        #    break
        if count % 1000 == 0:
            print(count)

        counts = grb['counts']
        spectrum = grb['spectrum']
        if "230" in spectrum :
            spectra_value="soft"
        elif "699.9" in spectrum:
            spectra_value="medium"
        elif "Compton" in spectrum:
            spectra_value="hard"
        else:
            spectra_value="random"

        theta_real = float(grb['coord'][0])
        phi_real = float(grb['coord'][1])

        # mean counts during a window of 500 seconds
        b_sim = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])
        global_min = -1
        if spectral_model=="soft":            
            
            chi_squared_soft = calculate_chi_squared_optimized(np.array(counts)+np.random.poisson(b_sim*20),b_sim*20, lut_soft)
            min_soft = np.min(chi_squared_soft)
            global_min = min_soft

            chi2_array = chi_squared_soft
            if spectra_value == "soft":
                spectra_fitted = spectra_fitted +1
                spectra_selected = True
            argmin_index = np.argmin(chi_squared_soft)
            best_lut = lut_soft
            
        elif spectral_model=="medium":
            
            chi_squared_medium = calculate_chi_squared_optimized(np.array(counts)+np.random.poisson(b_sim*20),b_sim*20, lut_medium)
            min_medium = np.min(chi_squared_medium)
            global_min = min_medium

            chi2_array = chi_squared_medium
            if spectra_value == "medium":
                spectra_fitted = spectra_fitted +1
                spectra_selected = True
            argmin_index = np.argmin(chi_squared_medium)
            best_lut = lut_medium
            
        elif spectral_model=="hard":
            
            chi_squared_hard = calculate_chi_squared_optimized(np.array(counts)+np.random.poisson(b_sim*20),b_sim*20, lut_hard)
            min_hard = np.min(chi_squared_hard)
            global_min = min_hard

            chi2_array = chi_squared_hard
            if spectra_value == "hard":
                spectra_fitted = spectra_fitted +1
                spectra_selected = True
            argmin_index = np.argmin(chi_squared_hard)
            best_lut = lut_hard

        
        if(global_min==-1):
            print("error")
       
        spectra_selected = False
                
        theta_loc = best_lut[argmin_index][6]
        phi_loc = best_lut[argmin_index][7]
    
        theta_real = float(grb['coord'][0])
        phi_real = float(grb['coord'][1])
    
        min_idx = find_closest_pixel(theta_real,phi_real,best_lut)
    
        real_chi2 = chi2_array[min_idx]
    
        dist = angular_distance(theta_loc,phi_loc,theta_real,phi_real)
        theta_dist = np.abs(theta_loc-theta_real)
        phi_dist = diff_phi(phi_loc,phi_real)
    
        if spectra_selected:
            good_spectra_fit_distances.append(dist)
        else:
            bad_spectra_fit_distances.append(dist)
    
        results.append([theta_real,phi_real,theta_loc,phi_loc,dist,theta_dist,phi_dist,global_min,real_chi2,chi2_array])
        
        count = count +1

    return results,spectra_fitted, good_spectra_fit_distances,bad_spectra_fit_distances

In [ ]:
results,spectra_fitted, good_spectra_fit_distances,bad_spectra_fit_distances= analyze_grbs(test_dataset,"soft")

In [ ]:
from scipy.stats import chi2
import healpy as hp

distances = []
theta_distances = []
phi_distances = []
areas_90 = []
for res in results:
    distances.append(res[4])
    theta_distances.append(res[5])
    phi_distances.append(res[6])

    # calculate area 
    map = res[9]

    limit = res[7]+ chi2.ppf(0.9, 2)
    
    # Maschera i valori maggiori o uguali a X
    nside = hp.npix2nside(len(map))
    
    mask_inside_region = map < limit
    npix_in_region = np.sum(mask_inside_region)
    nside = hp.get_nside(map)
    pix_area = hp.nside2pixarea(nside, degrees=True)
    area_90 = npix_in_region * pix_area
    areas_90.append(area_90)





In [ ]:
import pickle
if False:

    # Salva l'array in un file usando pickle
    with open(prefix+"/data/chi2_"+file_name_test+"_distall.pkl", "wb") as f:
        pickle.dump(distances, f)
        
    # Salva l'array in un file usando pickle
    with open(prefix+"/data/chi2_"+file_name_test+"_cont_area.pkl", "wb") as f:
        pickle.dump(areas_90, f)



In [ ]:
# Choose the index to plot. Eventually it is possible to implement a for cycle.
index = 0
min_chi2 = results[index][7]
theta_real = float(results[index][0])
phi_real = float(results[index][1])
theta_reco=float(results[index][2])
phi_reco=float(results[index][3])

if phi_real>180:
    phi_real = phi_real-360
if phi_reco>180:
    phi_reco = phi_reco-360

index = 0
map = results[index][9]

limit = results[index][7]+ chi2.ppf(0.9, 2)

# Maschera i valori maggiori o uguali a X
mappa_masked = np.ma.masked_where(map  >= limit , map)

hp.projview(
    mappa_masked,
    coord=["G"],
    projection_type="aitoff",          
    graticule=True,
    graticule_labels=True,
    longitude_grid_spacing=60,
    title=file,
    latitude_grid_spacing=30,
    cmap="viridis",
    nest=True,
    unit="", 
    badcolor="antiquewhite",
    fontsize={
        "xlabel": 14,
        "ylabel": 14,
        "title": 16,
        "xtick_label": 14,
        "ytick_label": 14,
        "cbar_label": 14,
        "cbar_tick_label": 14  # qui imposti il font size dei numeri della colorbar
    },
    override_plot_properties={
        "cbar_shrink": 0.8,
        "cbar_pad": 0.05,
        "cbar_label_pad": 0
    }
    
)

hp.newprojplot(theta=np.radians(theta_real), phi=np.radians(phi_real), marker="*", color="magenta", markersize=13)

hp.newprojplot(theta=np.radians(theta_reco), phi=np.radians(phi_reco), marker="X", color="r", markersize=12)

ax = plt.gca()

fig = plt.gcf()
axes = fig.get_axes()
main_ax = plt.gca()

for ax in axes:
    if ax != main_ax:
        cbar_ax = ax
        break

cbar_ax.set_xlabel("$\chi^2$ value (90% c.l.)", fontsize=14) 

plt.show()

In [ ]:

from scipy.stats import chi2
values = np.arange(0, 1.01, 0.01)

fraction_array = []

for value in values:
    count = 0 
    for r in results:
        min_chi2 = r[7]
        real_chi2 = r[8]
        if real_chi2<min_chi2+chi2.ppf(value, 2).flatten():
            count = count + 1
    fraction_array.append(count/len(results))
    


In [ ]:
plt.plot(values, fraction_array, label='Fraction')  # Etichetta della curva principale

# Linea unitaria
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Unitary Line')

plt.xlabel('Confidence Level')  # Nome dell'asse X
plt.ylabel('Fraction < Confidence Level')  # Nome dell'asse Y
plt.title('')  # (Opzionale) titolo del grafico
plt.legend()  # Mostra la legenda con le label definite
plt.grid(True)  # (Opzionale) griglia per migliorare la leggibilità
plt.show()

In [ ]:
fraction_array[90]

In [ ]:
import numpy as np
from scipy.stats import chi2
import matplotlib.pyplot as plt

# Confidence level threshold (fixed)
delta_chi2_90 = chi2.ppf(0.9, df=2)

# Define flux bins (adjust range and step as needed)
flux_bins = np.arange(0, 30 + 1, 1)
flux_centers = (flux_bins[:-1] + flux_bins[1:]) / 2
coverage_array = []

# For each flux bin
for i in range(len(flux_bins) - 1):
    count_in = 0
    count_total = 0
    count = 0
    for r in results:
        flux = float(test_dataset[count]['flux'])
        if flux_bins[i] <= flux < flux_bins[i+1]:
            min_chi2 = r[7]
            real_chi2 = r[8]
            if real_chi2 < min_chi2 + delta_chi2_90:
                count_in += 1
            count_total += 1
        count = count+1
        
    if count_total > 0:
        coverage = count_in / count_total
    else:
        coverage =  0
    coverage_array.append(coverage)

# Plotting
plt.plot(flux_centers, coverage_array, marker='o')
plt.axhline(0.9, color='gray', linestyle='--', label='Target 90%')
plt.xlabel('Source Flux')
plt.ylabel('Empirical 90% Coverage')
plt.title('Coverage vs. Source Flux at Fixed 90% CL')
plt.legend()
plt.grid(True)
plt.show()
